# 🖼️ minicv — Complete API Demo Notebook

> **Course demo** — This notebook exercises every public function in the
> `minicv` library on a single RGB input image, with short explanations
> of what each operation does and how to interpret the output.

**Library modules covered:**
| Module | Contents |
|--------|----------|
| `minicv.utils` | Normalisation · Clipping · Padding · Convolution |
| `minicv.io` | Read · Write · Color conversion |
| `minicv.filtering` | Blur · Threshold · Sobel · Canny · Histogram · Segmentation |
| `minicv.transforms` | Resize · Rotate · Translate |
| `minicv.features` | Color histogram · Pixel statistics · HOG · LBP |
| `minicv.drawing` | Point · Line · Rectangle · Polygon · Text |


---
## 0 · Environment Setup

Import all library modules and configure Matplotlib for inline display.


In [ ]:
import sys, os
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# ── Make sure the minicv package folder is on the path ──
# Adjust this path if your folder layout differs.
MINICV_ROOT = os.path.abspath('.')   # folder that contains minicv/
if MINICV_ROOT not in sys.path:
    sys.path.insert(0, MINICV_ROOT)

import minicv.utils      as utils
import minicv.io         as io
import minicv.filtering  as filt
import minicv.transforms as transforms
import minicv.features   as features
import minicv.drawing    as drawing

print('minicv imported successfully ✓')
print('NumPy  :', np.__version__)
print('Matplotlib:', matplotlib.__version__)


In [ ]:
# ── Matplotlib display settings ──
%matplotlib inline
plt.rcParams.update({
    'figure.dpi'      : 110,
    'axes.titlesize'  : 10,
    'axes.titleweight': 'bold',
    'figure.facecolor': 'white',
})

# Utility: show a row of images with titles
def show_row(images, titles, cmap_list=None, figw=18, figh=3.8):
    n = len(images)
    fig, axes = plt.subplots(1, n, figsize=(figw, figh))
    if n == 1: axes = [axes]
    for ax, img, title, cmap in zip(axes, images, titles,
                                    cmap_list or ['gray']*n):
        if img.ndim == 3:
            ax.imshow(img)
        else:
            vmax = 255 if img.dtype == np.uint8 else None
            ax.imshow(img, cmap=cmap, vmin=0, vmax=vmax)
        ax.set_title(title)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

def show_two(a, ta, b, tb, cma='gray', cmb='gray'):
    show_row([a,b],[ta,tb],[cma,cmb], figw=8, figh=3.5)

print('Helpers defined ✓')


---
## 1 · Load the Sample Image

We use `minicv.io.read_image` to load any PNG or JPEG from disk into a
`numpy.ndarray` of shape **(H, W, 3)** with dtype `uint8`.

> **To use your own image:** change `IMAGE_PATH` to any `.png` or `.jpg` file.
> The rest of the notebook adapts automatically.


In [ ]:
# ── Change this path to any PNG/JPG on your machine ──
IMAGE_PATH = 'sample.png'

img  = io.read_image(IMAGE_PATH)           # RGB  uint8  (H, W, 3)
gray = io.read_image(IMAGE_PATH, as_gray=True)  # gray uint8  (H, W)

print(f'RGB  image: shape={img.shape}   dtype={img.dtype}'
      f'   min={img.min()}  max={img.max()}')
print(f'Gray image: shape={gray.shape}  dtype={gray.dtype}'
      f'   min={gray.min()}  max={gray.max()}')


In [ ]:
show_row([img, gray],
         ['Original RGB  (H×W×3 uint8)', 'Grayscale  (H×W uint8)'],
         [None, 'gray'], figw=10, figh=4.5)


---
## 2 · I/O & Color Conversion

### 2.1 `rgb_to_gray` — Luminance-weighted grayscale
Uses the ITU-R BT.601 formula:
$$Y = 0.299 \cdot R + 0.587 \cdot G + 0.114 \cdot B$$

### 2.2 `gray_to_rgb` — Broadcast gray to 3 channels
Stacks the grayscale array three times along a new axis,
producing a neutral-grey RGB image.

### 2.3 `write_image` — Export to disk
Saves any `uint8` array as PNG or JPEG.


In [ ]:
gray_from_rgb = io.rgb_to_gray(img)       # (H, W) uint8
rgb_from_gray = io.gray_to_rgb(gray)      # (H, W, 3) uint8

print('rgb_to_gray output shape:', gray_from_rgb.shape)
print('gray_to_rgb output shape:', rgb_from_gray.shape)

show_row([img, gray_from_rgb, rgb_from_gray],
         ['Original RGB', 'rgb_to_gray', 'gray_to_rgb (neutral)'],
         [None, 'gray', None])


In [ ]:
import tempfile, os
with tempfile.TemporaryDirectory() as tmpdir:
    out_png = os.path.join(tmpdir, 'exported.png')
    io.write_image(img, out_png)
    reloaded = io.read_image(out_png)

print('write_image → read_image roundtrip max diff:',
      np.abs(img.astype(int) - reloaded.astype(int)).max())
print('(0 = perfectly lossless for PNG)')


---
## 3 · Core Utilities


### 3.1 Normalisation — three modes

| Mode | Formula | Typical use |
|------|---------|-------------|
| `minmax` | $x' = \frac{x - x_{min}}{x_{max}-x_{min}}(high-low)+low$ | display, rescaling |
| `zscore` | $x' = (x-\mu)/\sigma$ | ML feature prep |
| `fixed`  | $x' = x / high$ | fast uint8→float |


In [ ]:
norm_minmax = utils.normalize(gray, mode='minmax', low=0, high=255)  # uint8 out
norm_zscore = utils.normalize(gray, mode='zscore')                   # float64 out
norm_fixed  = utils.normalize(gray, mode='fixed',  high=255)        # float64 out

print('minmax dtype:', norm_minmax.dtype, ' range:', norm_minmax.min(), '-', norm_minmax.max())
print('zscore dtype:', norm_zscore.dtype, ' mean ≈', round(norm_zscore.mean(),3),
      ' std ≈', round(norm_zscore.std(),3))
print('fixed  dtype:', norm_fixed.dtype,  ' range:', norm_fixed.min(), '-', round(norm_fixed.max(),3))

show_row([gray, norm_minmax, norm_zscore, norm_fixed],
         ['Original gray', 'normalize minmax', 'normalize zscore', 'normalize fixed'],
         ['gray','gray','RdBu_r','gray'])


### 3.2 Pixel Clipping

Clamps every pixel to the interval $[low,\ high]$.
$$\text{clip}(x, l, h) = \max(l,\ \min(x, h))$$


In [ ]:
clipped = utils.clip_pixels(gray, low=80, high=200)
print('Original  range:', gray.min(), '-', gray.max())
print('Clipped   range:', clipped.min(), '-', clipped.max())
show_two(gray, 'Original gray', clipped, 'clip_pixels [80, 200]')


### 3.3 Padding — three modes

Adds a border of width `pad_h` × `pad_w` around the image.
Used internally before every convolution to maintain output size.

| Mode | Border rule |
|------|-------------|
| `reflect` | Mirror: `dcb│abcde│dcb` |
| `constant` | Fill with a fixed value (default 0) |
| `replicate` | Repeat edge pixels: `aaa│abcde│eee` |


In [ ]:
pad_reflect   = utils.pad_image(gray, 30, 30, mode='reflect')
pad_constant  = utils.pad_image(gray, 30, 30, mode='constant',  constant_value=128)
pad_replicate = utils.pad_image(gray, 30, 30, mode='replicate')

print('Original shape:', gray.shape)
print('Padded   shape:', pad_reflect.shape, '(+60 px each dim)')

show_row([pad_reflect, pad_constant, pad_replicate],
         ['pad reflect', 'pad constant (=128)', 'pad replicate'],
         ['gray','gray','gray'])


### 3.4 2D Convolution (`convolve2d`)

True 2D convolution (kernel flipped 180°) implemented via NumPy stride-tricks —
no Python pixel loops.

$$(f \star g)[m,n] = \sum_{i}\sum_{j} f[i,j]\cdot g[m-i,\,n-j]$$

Below we demonstrate with an **identity kernel** (output = input),
a **sharpen kernel**, and a **horizontal edge kernel**.


In [ ]:
img_f = gray.astype(np.float64)   # convolve2d requires float / single channel

identity = np.array([[0,0,0],[0,1,0],[0,0,0]], dtype=np.float64)
sharpen  = np.array([[ 0,-1, 0],[-1, 5,-1],[ 0,-1, 0]], dtype=np.float64)
horiz    = np.array([[-1,-1,-1],[0,0,0],[1,1,1]], dtype=np.float64)

out_identity = utils.convolve2d(img_f, identity)
out_sharpen  = utils.convolve2d(img_f, sharpen)
out_horiz    = utils.convolve2d(img_f, horiz)

print('Identity max diff from original:', np.abs(out_identity - img_f).max())

show_row([gray,
          np.clip(out_sharpen, 0, 255).astype(np.uint8),
          np.clip(np.abs(out_horiz), 0, 255).astype(np.uint8)],
         ['Original', 'convolve2d: sharpen kernel', 'convolve2d: horizontal edges'],
         ['gray','gray','gray'])


### 3.5 Spatial Filter (`spatial_filter`)

`spatial_filter` applies any 2D kernel to **grayscale or RGB** images.
For RGB it convolves each channel independently (per-channel strategy).


In [ ]:
emboss = np.array([[-2,-1,0],[-1,1,1],[0,1,2]], dtype=np.float64)

filtered_gray = utils.spatial_filter(gray, emboss)  # uint8 out
filtered_rgb  = utils.spatial_filter(img,  emboss)  # uint8 out

print('spatial_filter gray output:', filtered_gray.shape, filtered_gray.dtype)
print('spatial_filter RGB  output:', filtered_rgb.shape,  filtered_rgb.dtype)

show_row([gray, filtered_gray, img, filtered_rgb],
         ['Gray original', 'spatial_filter gray (emboss)',
          'RGB original',  'spatial_filter RGB (emboss)'],
         ['gray','gray', None, None])


---
## 4 · Image Processing Techniques


### 4.1 Mean / Box Filter

Replaces each pixel with the uniform average of its $k\times k$ neighbourhood.
$$h[m,n] = \frac{1}{k^2}\sum_{(i,j)\in\text{window}} f[i,j]$$
Larger kernels → stronger blur, loss of detail.


In [ ]:
mean3  = filt.mean_filter(img, kernel_size=3)
mean9  = filt.mean_filter(img, kernel_size=9)
mean15 = filt.mean_filter(img, kernel_size=15)

show_row([img, mean3, mean9, mean15],
         ['Original', 'mean k=3', 'mean k=9', 'mean k=15'])


### 4.2 Gaussian Filter

Weights neighbours by a 2D Gaussian — pixels closer to the centre contribute more.
$$G(x,y) = \frac{1}{2\pi\sigma^2}\exp\!\left(-\frac{x^2+y^2}{2\sigma^2}\right)$$
Preserves edges better than the box filter at equivalent kernel sizes.


In [ ]:
# Inspect the kernel shape first
k55 = filt.gaussian_kernel(size=5, sigma=1.0)
print('5×5 Gaussian kernel (σ=1.0):')
print(np.round(k55, 4))
print('Kernel sum:', round(k55.sum(), 6))


In [ ]:
gauss_s1 = filt.gaussian_filter(img, kernel_size=5,  sigma=1.0)
gauss_s2 = filt.gaussian_filter(img, kernel_size=9,  sigma=2.0)
gauss_s4 = filt.gaussian_filter(img, kernel_size=15, sigma=4.0)

show_row([img, gauss_s1, gauss_s2, gauss_s4],
         ['Original', 'gaussian k=5 σ=1', 'gaussian k=9 σ=2', 'gaussian k=15 σ=4'])


### 4.3 Median Filter

Replaces each pixel with the **median** of its neighbourhood — a non-linear
operation that removes salt-and-pepper noise while preserving sharp edges.

$$h[m,n] = \text{median}\{\,f[i,j]\;:\;(i,j)\in k\times k\text{ window}\}$$

> **Note:** To illustrate median's strength, we first add artificial noise.


In [ ]:
# Add salt-and-pepper noise to demonstrate median's advantage
rng = np.random.default_rng(42)
noisy = img.copy()
n_salt = int(0.03 * img.size)
coords = [rng.integers(0, s, n_salt) for s in img.shape]
noisy[coords[0], coords[1]] = 255   # salt
coords2 = [rng.integers(0, s, n_salt) for s in img.shape]
noisy[coords2[0], coords2[1]] = 0   # pepper

med3 = filt.median_filter(noisy, kernel_size=3)
med5 = filt.median_filter(noisy, kernel_size=5)
mean_ref = filt.mean_filter(noisy, kernel_size=3)

show_row([noisy, mean_ref, med3, med5],
         ['Noisy (3% s&p)', 'mean k=3 (blurs noise+edges)',
          'median k=3 (clean edges)', 'median k=5'])


### 4.4 Thresholding

Converts a grayscale image to binary by comparing each pixel to a threshold $T$.

| Method | How $T$ is chosen |
|--------|-------------------|
| **Global** | User-specified constant |
| **Otsu** | Maximise inter-class variance automatically |
| **Adaptive** | Local neighbourhood mean or Gaussian-weighted mean |


In [ ]:
# Global
thresh_global = filt.threshold_global(gray, thresh=128)
thresh_inv    = filt.threshold_global(gray, thresh=128, invert=True)

# Otsu
thresh_otsu, t_val = filt.threshold_otsu(gray)
print(f'Otsu optimal threshold: {t_val:.1f}')

# Adaptive
thresh_adapt_mean = filt.threshold_adaptive(gray, block_size=31, method='mean',     C=5)
thresh_adapt_gaus = filt.threshold_adaptive(gray, block_size=31, method='gaussian', C=5)

show_row([gray, thresh_global, thresh_inv, thresh_otsu,
          thresh_adapt_mean, thresh_adapt_gaus],
         ['Gray', 'global T=128', 'global inverted',
          f'Otsu (T={t_val:.0f})', 'adaptive mean', 'adaptive gaussian'],
         ['gray']*6, figw=22, figh=4)


### 4.5 Sobel Gradients

Approximates the image gradient using two 3×3 kernels:
$$G_x = \begin{bmatrix}-1&0&1\\-2&0&2\\-1&0&1\end{bmatrix}\!,\quad
G_y = \begin{bmatrix}-1&-2&-1\\0&0&0\\1&2&1\end{bmatrix}$$
$$|G| = \sqrt{G_x^2 + G_y^2},\qquad \theta = \arctan(G_y/G_x) \bmod 180°$$


In [ ]:
grads = filt.sobel_gradients(gray)
Gx, Gy  = grads['Gx'], grads['Gy']
mag, ang = grads['magnitude'], grads['angle']

print('Gradient magnitude — min:', round(mag.min(),2), ' max:', round(mag.max(),2))
print('Angle range [0°, 180°)  — min:', round(ang.min(),2), ' max:', round(ang.max(),2))

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, arr, title, cmap in zip(axes,
        [Gx, Gy, mag, ang],
        ['Gx (horizontal)', 'Gy (vertical)', '|G| magnitude', 'θ angle (°)'],
        ['RdBu_r','RdBu_r','hot','hsv']):
    im = ax.imshow(arr, cmap=cmap)
    ax.set_title(title); ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout(); plt.show()


### 4.6 Bit-Plane Slicing

Every uint8 pixel is composed of 8 bit positions (planes 0–7).
Isolating plane $n$ reveals which pixels have the $2^n$ bit set:
$$BP_n(x,y) = \begin{cases}255 & \text{if } f(x,y) \mathbin{\&} 2^n > 0 \\ 0 & \text{otherwise}\end{cases}$$
High-order planes (6, 7) carry dominant structure; low-order planes (0, 1) carry noise.


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for n in range(8):
    ax = axes[n // 4][n % 4]
    bp = filt.bit_plane_slice(gray, plane=n)
    ax.imshow(bp, cmap='gray', vmin=0, vmax=255)
    ax.set_title(f'Plane {n}  (2^{n} = {2**n})')
    ax.axis('off')
plt.suptitle('Bit-Plane Slicing (plane 0 = LSB, plane 7 = MSB)', fontsize=12)
plt.tight_layout(); plt.show()


### 4.7 Histogram & Histogram Equalization

**Histogram** counts pixels per intensity level:
$$h(k) = |\{(x,y) : f(x,y)=k\}|, \quad p(k) = h(k)/N$$

**Equalization** maps intensities through the CDF to spread them uniformly:
$$T(k) = \text{round}\!\left((L-1)\cdot\frac{\text{CDF}(k)-\text{CDF}_{\min}}{1-\text{CDF}_{\min}}\right)$$


In [ ]:
counts, edges = filt.histogram(gray, bins=256)
equalized     = filt.histogram_equalization(gray)
eq_counts, _  = filt.histogram(equalized, bins=256)

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

axes[0,0].imshow(gray, cmap='gray', vmin=0, vmax=255)
axes[0,0].set_title('Original grayscale'); axes[0,0].axis('off')

axes[0,1].imshow(equalized, cmap='gray', vmin=0, vmax=255)
axes[0,1].set_title('After histogram equalization'); axes[0,1].axis('off')

bcentres = (edges[:-1] + edges[1:]) / 2
axes[1,0].bar(bcentres, counts, width=1, color='steelblue', alpha=0.85)
axes[1,0].set_title('Histogram — original')
axes[1,0].set_xlabel('Intensity'); axes[1,0].set_ylabel('Count')

axes[1,1].bar(bcentres, eq_counts, width=1, color='darkorange', alpha=0.85)
axes[1,1].set_title('Histogram — after equalization')
axes[1,1].set_xlabel('Intensity'); axes[1,1].set_ylabel('Count')

plt.tight_layout(); plt.show()


### 4.8 Canny Edge Detection

A four-stage pipeline:
1. **Gaussian smoothing** — suppress noise
2. **Sobel gradients** — estimate edge strength & direction
3. **Non-maximum suppression** — thin ridges to 1-pixel width
4. **Hysteresis thresholding** — keep only strong edges and weak edges connected to them


In [ ]:
edges_fine   = filt.canny(gray, low_threshold=20,  high_threshold=60)
edges_medium = filt.canny(gray, low_threshold=50,  high_threshold=130)
edges_strict = filt.canny(gray, low_threshold=100, high_threshold=220)

print('Edge pixels (medium):', edges_medium.sum() // 255)

show_row([gray, edges_fine, edges_medium, edges_strict],
         ['Gray', 'Canny lo=20 hi=60', 'Canny lo=50 hi=130', 'Canny lo=100 hi=220'],
         ['gray','gray','gray','gray'])


### 4.9 K-Means Clustering (Segmentation)

Groups pixels into $k$ clusters by minimising within-cluster variance.
Each pixel is replaced by its cluster's centroid colour.

$$\min \sum_{c=1}^{k}\sum_{\mathbf{x}_i\in C_c}\|\mathbf{x}_i - \boldsymbol{\mu}_c\|^2$$

Initialisation uses **K-Means++** seeding for faster convergence.


In [ ]:
seg2, lbl2 = filt.kmeans_segment(img, k=2, max_iter=50)
seg4, lbl4 = filt.kmeans_segment(img, k=4, max_iter=50)
seg6, lbl6 = filt.kmeans_segment(img, k=6, max_iter=50)

print('Unique labels (k=4):', np.unique(lbl4))

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, arr, title in zip(axes.flat,
        [img, seg2, seg4, seg6, lbl4, lbl6],
        ['Original RGB','k=2 segmented','k=4 segmented','k=6 segmented',
         'k=4 label map','k=6 label map']):
    cmap = 'tab10' if 'label' in title else None
    ax.imshow(arr, cmap=cmap); ax.set_title(title); ax.axis('off')
plt.tight_layout(); plt.show()


---
## 5 · Geometric Transformations

All transforms use **inverse mapping**: for each destination pixel $(r_d,c_d)$
we compute the source coordinate $(r_s,c_s)$ and sample the source image.
This avoids holes in the output.


### 5.1 Resize

**Nearest-neighbour** rounds $(r_s, c_s)$ to the nearest integer.
**Bilinear** interpolates from the four surrounding source pixels:
$$f' = (1-\Delta r)(1-\Delta c)f_{00} + (1-\Delta r)\Delta c\,f_{01}
+ \Delta r(1-\Delta c)f_{10} + \Delta r\,\Delta c\,f_{11}$$


In [ ]:
H, W = img.shape[:2]

tiny_nn  = transforms.resize(img, H//4, W//4, interpolation='nearest')
tiny_bi  = transforms.resize(img, H//4, W//4, interpolation='bilinear')
large_bi = transforms.resize(img, H*2,  W*2,  interpolation='bilinear')

print('Down (nearest):', tiny_nn.shape)
print('Down (bilinear):', tiny_bi.shape)
print('Up   (bilinear):', large_bi.shape)

# Zoom-in patch to see interpolation difference
patch_nn = transforms.resize(tiny_nn[20:40, 20:40], 160, 160, 'nearest')
patch_bi = transforms.resize(tiny_bi[20:40, 20:40], 160, 160, 'nearest')

show_row([tiny_nn, tiny_bi, patch_nn, patch_bi],
         ['Downscale ÷4 nearest', 'Downscale ÷4 bilinear',
          'Patch ×8 nearest (blocky)', 'Patch ×8 bilinear (smooth)'])


### 5.2 Rotation

Rotates about the image centre using the inverse rotation matrix:
$$\begin{pmatrix}r_s - c_y \\ c_s - c_x\end{pmatrix} =
\begin{pmatrix}\cos\alpha & \sin\alpha \\ -\sin\alpha & \cos\alpha\end{pmatrix}
\begin{pmatrix}r_d - c_y \\ c_d - c_x\end{pmatrix}$$
Pixels mapping outside the canvas are filled with 0 (black).


In [ ]:
rot15  = transforms.rotate(img,  15, interpolation='bilinear')
rot45  = transforms.rotate(img,  45, interpolation='bilinear')
rot90  = transforms.rotate(img,  90, interpolation='bilinear')
rot180 = transforms.rotate(img, 180, interpolation='bilinear')

show_row([img, rot15, rot45, rot90, rot180],
         ['Original', 'rotate +15°', 'rotate +45°', 'rotate +90°', 'rotate +180°'],
         figw=22, figh=4)


### 5.3 Translation

Shifts the image by $(t_x, t_y)$ pixels using inverse mapping:
$$r_s = r_d - t_y, \quad c_s = c_d - t_x$$


In [ ]:
trans_r  = transforms.translate(img,  60,   0)   # right
trans_d  = transforms.translate(img,   0,  60)   # down
trans_dl = transforms.translate(img, -80,  80)   # down-left

show_row([img, trans_r, trans_d, trans_dl],
         ['Original', 'translate →60px', 'translate ↓60px', 'translate ←80 ↓80'])


---
## 6 · Feature Descriptors


### 6.1 Global: Color Histogram

Counts pixels per intensity bin per channel, then L1-normalises.
Result: a 1D vector of length `n_bins` (gray) or `3 × n_bins` (RGB).
Useful for image retrieval and colour-based classification.


In [ ]:
h_gray = features.color_histogram(gray, bins=32, normalize=True)
h_rgb  = features.color_histogram(img,  bins=32, normalize=True)

print('Gray histogram shape:', h_gray.shape, '  sum:', round(h_gray.sum(),4))
print('RGB  histogram shape:', h_rgb.shape,  '  (3×32 bins)')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
x32 = np.arange(32)
axes[0].bar(x32, h_gray, color='dimgray', alpha=0.8)
axes[0].set_title('Grayscale color_histogram (32 bins)')
axes[0].set_xlabel('Bin'); axes[0].set_ylabel('Normalised frequency')

colours = ['#e74c3c','#27ae60','#2980b9']
for c, col in enumerate(colours):
    axes[1].bar(x32 + c*0.28, h_rgb[c*32:(c+1)*32], width=0.28,
                color=col, alpha=0.8, label='RGB'[c])
axes[1].set_title('RGB color_histogram (32 bins per channel)')
axes[1].set_xlabel('Bin'); axes[1].legend()
plt.tight_layout(); plt.show()


### 6.2 Global: Pixel Statistics

Computes per-channel descriptive statistics as a dictionary:

| Statistic | Formula |
|-----------|---------|
| Mean | $\mu = \frac{1}{N}\sum x_i$ |
| Std Dev | $\sigma = \sqrt{\frac{1}{N}\sum(x_i-\mu)^2}$ |
| Skewness | $\gamma_1 = E[(X-\mu)^3]/\sigma^3$ |
| Kurtosis | $\gamma_2 = E[(X-\mu)^4]/\sigma^4 - 3$ |
| Entropy | $H = -\sum p(k)\log_2 p(k)$ |


In [ ]:
stats = features.pixel_statistics(img)

import pandas as pd
rows = []
stat_names = ['mean','std','skewness','kurtosis','energy','entropy','min','max']
for ch, name in enumerate(['Red','Green','Blue']):
    row = {'Channel': name}
    for s in stat_names:
        key = f'{s}_{ch}'
        row[s.capitalize()] = round(stats[key], 4) if key in stats else '-'
    rows.append(row)

df = pd.DataFrame(rows).set_index('Channel')
print('pixel_statistics — RGB image:')
print(df.to_string())


### 6.3 Gradient: HOG Descriptor

**Histogram of Oriented Gradients** — captures local edge orientations:
1. Compute Sobel magnitude & angle per pixel
2. Soft-bin each pixel's gradient into two adjacent orientation bins
3. Accumulate into cells (e.g. 8×8 px), form blocks (e.g. 2×2 cells)
4. **L2-Hys** normalise each block: $v \leftarrow v/\|v\|_2,\ $ clip at 0.2,$\ $ renormalise


In [ ]:
# HOG on grayscale
hog_vec = features.hog_descriptor(gray, cell_size=8, n_bins=9, block_size=2)
print(f'HOG descriptor length: {len(hog_vec)}')
print(f'Value range: [{hog_vec.min():.4f}, {hog_vec.max():.4f}]')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].imshow(gray, cmap='gray'); axes[0].set_title('Input gray image'); axes[0].axis('off')
axes[1].plot(hog_vec, color='#2980b9', linewidth=0.6)
axes[1].fill_between(range(len(hog_vec)), hog_vec, alpha=0.3, color='#2980b9')
axes[1].set_title(f'HOG descriptor (length={len(hog_vec)})')
axes[1].set_xlabel('Feature index'); axes[1].set_ylabel('Value')
plt.tight_layout(); plt.show()


### 6.4 Gradient: LBP Descriptor

**Local Binary Pattern** — encodes micro-texture by comparing each pixel
to its 8 neighbours, forming an 8-bit code:
$$\text{LBP}(x,y) = \sum_{n=0}^{7} s(g_n - g_c)\cdot 2^n, \quad s(u)=\begin{cases}1&u\geq0\\0&u<0\end{cases}$$
The descriptor is the normalised histogram of LBP codes (256 possible values).


In [ ]:
lbp_vec = features.lbp_descriptor(gray, n_bins=256, normalize=True)
print(f'LBP descriptor length: {len(lbp_vec)}  sum: {lbp_vec.sum():.4f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].imshow(gray, cmap='gray'); axes[0].set_title('Input gray image'); axes[0].axis('off')
axes[1].bar(range(256), lbp_vec, width=1.0, color='#8e44ad', alpha=0.8)
axes[1].set_title('LBP histogram (256 bins)')
axes[1].set_xlabel('LBP code (0–255)'); axes[1].set_ylabel('Normalised frequency')
plt.tight_layout(); plt.show()


---
## 7 · Drawing Primitives & 8 · Text

All drawing functions work **in-place** on uint8 NumPy arrays.
Coordinates outside the canvas are silently clipped.
Color format: **scalar** for grayscale, **RGB 3-tuple** for colour.


### 7.1 `draw_point` — Filled circle

Uses a vectorised disk mask:
$$\text{mask} = \{(r,c) : \sqrt{(r-y)^2+(c-x)^2}\leq\text{radius}\}$$


In [ ]:
canvas = img.copy()

drawing.draw_point(canvas, x=80,  y=80,  color=(255,50,50),  radius=20)
drawing.draw_point(canvas, x=200, y=60,  color=(50,255,50),  radius=14)
drawing.draw_point(canvas, x=340, y=100, color=(50,100,255), radius=25)
drawing.draw_point(canvas, x=450, y=70,  color=(255,200,0),  radius=10)

show_two(img, 'Original', canvas, 'draw_point (various radii & colours)', cmb=None)


### 7.2 `draw_line` — Bresenham's algorithm

Integer-arithmetic rasterisation — decides each step which pixel is closer
to the ideal mathematical line.


In [ ]:
canvas = img.copy()
H2, W2 = canvas.shape[:2]

# Grid of diagonal lines
drawing.draw_line(canvas, 0, 0, W2-1, H2-1, (255,255,0), thickness=3)
drawing.draw_line(canvas, W2-1, 0, 0, H2-1, (0,255,255), thickness=3)
drawing.draw_line(canvas, 0, H2//2, W2-1, H2//2, (255,80,80), thickness=2)
drawing.draw_line(canvas, W2//2, 0, W2//2, H2-1, (80,255,80), thickness=2)
drawing.draw_line(canvas, 50, 10, 300, 200, (255,160,0), thickness=4)

show_two(img, 'Original', canvas, 'draw_line (Bresenham)', cmb=None)


### 7.3 `draw_rectangle` — Outline & filled


In [ ]:
canvas = img.copy()

# Outlines
drawing.draw_rectangle(canvas, x=20,  y=20,  width=120, height=80,
                       color=(255,50,50),  thickness=3, filled=False)
drawing.draw_rectangle(canvas, x=160, y=20,  width=100, height=70,
                       color=(50,255,50),  thickness=2, filled=False)

# Filled
drawing.draw_rectangle(canvas, x=300, y=20,  width=90,  height=60,
                       color=(50,100,255), filled=True)
drawing.draw_rectangle(canvas, x=410, y=20,  width=80,  height=60,
                       color=(255,200,0),  filled=True)

show_two(img, 'Original', canvas, 'draw_rectangle: outline + filled', cmb=None)


### 7.4 `draw_polygon` — Outline & scanline fill

Filled mode uses **scanline rasterisation**: for each horizontal row
find edge intersections, sort them, fill between pairs.


In [ ]:
canvas = img.copy()

# Triangle outline
triangle = [(80, 400), (160, 280), (240, 400)]
drawing.draw_polygon(canvas, triangle, color=(255,255,0), thickness=3)

# Pentagon filled
import math
cx, cy, r = 370, 360, 70
pentagon = [(int(cx + r*math.cos(math.radians(90 - 72*i))),
             int(cy - r*math.sin(math.radians(90 - 72*i))))
            for i in range(5)]
drawing.draw_polygon(canvas, pentagon, color=(0,200,255), filled=True)

# Star outline
star_pts = []
for i in range(10):
    angle = math.radians(90 - 36*i)
    rad = 55 if i % 2 == 0 else 25
    star_pts.append((int(460 + rad*math.cos(angle)),
                     int(440 + rad*math.sin(angle))))
drawing.draw_polygon(canvas, star_pts, color=(255,80,80), thickness=2)

show_two(img, 'Original', canvas,
         'draw_polygon: triangle outline, pentagon fill, star outline', cmb=None)


### 8.1 `draw_text` — Matplotlib rasterisation + alpha composite

Renders Unicode text via Matplotlib, then alpha-composites onto the image:
$$\text{out}[c] = \alpha \cdot \text{color}[c] + (1-\alpha)\cdot\text{image}[c]$$


In [ ]:
canvas = img.copy()

drawing.draw_text(canvas, 'minicv Library', x=30,  y=20,
                  font_scale=1.8, color=(255,255,255), thickness=2)
drawing.draw_text(canvas, 'Milestone 1',    x=30,  y=70,
                  font_scale=1.2, color=(255,230,80))
drawing.draw_text(canvas, 'Grayscale value: 128', x=30, y=430,
                  font_scale=0.9, color=(200,255,200))
drawing.draw_text(canvas, 'numpy + matplotlib only',
                  x=30, y=470, font_scale=0.85, color=(180,220,255))

show_two(img, 'Original', canvas, 'draw_text (alpha composite)', cmb=None)


---
## 9 · Full Pipeline Showcase

Apply a realistic end-to-end processing pipeline on the original image:
denoise → edge-detect → annotate results.


In [ ]:
# Step 1: Denoise with Gaussian
denoised = filt.gaussian_filter(img, kernel_size=5, sigma=1.2)

# Step 2: Convert to gray
denoised_gray = io.rgb_to_gray(denoised)

# Step 3: Canny edges
edges = filt.canny(denoised_gray, low_threshold=40, high_threshold=110)

# Step 4: K-means segmentation
seg, labels = filt.kmeans_segment(img, k=5, max_iter=60)

# Step 5: Equalize contrast
eq = filt.histogram_equalization(io.rgb_to_gray(img))

# Step 6: Annotate
annotated = seg.copy()
drawing.draw_text(annotated, 'k=5 segmentation', x=10, y=10,
                  font_scale=1.1, color=(255,255,255), thickness=2)

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
specs = [
    (img,           'Original RGB',              None),
    (denoised,      'Step 1: Gaussian denoised', None),
    (denoised_gray, 'Step 2: Gray',              'gray'),
    (edges,         'Step 3: Canny edges',       'gray'),
    (eq,            'Step 5: Hist. equalized',   'gray'),
    (annotated,     'Step 4+6: Segmented + text',None),
]
for ax, (arr, title, cmap) in zip(axes.flat, specs):
    vmax = 255 if arr.dtype == np.uint8 else None
    ax.imshow(arr, cmap=cmap, vmin=0, vmax=vmax)
    ax.set_title(title, fontsize=11); ax.axis('off')
plt.suptitle('End-to-end Pipeline', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()


---
## ✅ Summary

This notebook demonstrated **every public function** in the `minicv` library:

| # | API | Functions demonstrated |
|---|-----|------------------------|
| 2 | `minicv.io` | `read_image`, `write_image`, `rgb_to_gray`, `gray_to_rgb` |
| 3 | `minicv.utils` | `normalize` (3 modes), `clip_pixels`, `pad_image` (3 modes), `convolve2d`, `spatial_filter` |
| 4 | `minicv.filtering` | `mean_filter`, `gaussian_kernel`, `gaussian_filter`, `median_filter`, `threshold_global`, `threshold_otsu`, `threshold_adaptive`, `sobel_gradients`, `bit_plane_slice`, `histogram`, `histogram_equalization`, `canny`, `kmeans_segment` |
| 5 | `minicv.transforms` | `resize` (nearest + bilinear), `rotate`, `translate` |
| 6 | `minicv.features` | `color_histogram`, `pixel_statistics`, `hog_descriptor`, `lbp_descriptor` |
| 7–8 | `minicv.drawing` | `draw_point`, `draw_line`, `draw_rectangle`, `draw_polygon`, `draw_text` |

**Built with:** NumPy · Matplotlib · Pandas · Python standard library — no OpenCV.
